# The whole project in 15 minutes


**EN** — This notebook compresses the whole study into nine steps you can run and
edit. It is not the course (that is `steps/01`–`steps/07`); it is the **story end to
end**: from "this strategy looks very profitable" to "it does not actually pick stocks".

---

### 怎么用

-  **Select Kernel** → **Python Environments** →  `.venv` 。
  <br>Top right → **Select Kernel** → **Python Environments** → the one inside `.venv`.
- `Shift + Enter` 逐格运行。
  <br>Clear all outputs before committing, or `git diff` becomes unreadable.


> <br>**Does 12-month momentum pick stocks, or does it just pick beta?**

In [ ]:
# --- 准备工作 / Setup -------------------------------------------------------
# notebook 没有 __file__，所以直接指明项目根目录。
# A notebook has no __file__, so we point at the project root explicitly.
import os, sys
ROOT = os.path.expanduser("~/tim")
'''
objective
1. learn python engineering
2. get familiar with the code base, packages, and modules
'''
os.chdir(ROOT)                                     # 让 "data/..." 找得到 / so "data/..." resolves
sys.path[:0] = [ROOT, os.path.join(ROOT, "lib")]   # config.py 在根目录，模块在 lib/

import math
import numpy as np
import pandas as pd

import analysis, config, engine, timlib as tl
from strategies import STRATEGIES, BY_ID

# timlib 在 import 时把 matplotlib 设成了 Agg（画到文件，因为服务器没有屏幕）。
# 在 notebook 里我们要图**显示出来**，所以这一行必须放在 import timlib 之后。
# timlib sets the Agg backend on import (draw to a file — the server has no screen).
# In a notebook we want the figures shown, so this magic must come AFTER that import.
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9, 4)

print(f"pandas {pd.__version__}   numpy {np.__version__}")

In [ ]:
ROOT

---
## 1 · 数据

<br>109 CSVs: the S&P-100 plus 9 factor ETFs, daily, Jan 2015 – Jul 2026.

<br>This is *today's* S&P-100 backfilled. Everything dropped, acquired or bankrupted in
between is missing, which inflates every return number below.

In [ ]:
daily = tl.load_prices()                                     # 日线收盘价 / daily closes
px    = tl.to_monthly(daily)                                 # 月末价格 / month-end prices
vol   = tl.to_monthly(tl.load_prices(column="Volume"), how="mean")
dates = list(px.index)

print(f"日线 daily   : {daily.shape}   (交易日 days, 股票 tickers)")
print(f"月线 monthly : {px.shape}      {dates[0].date()} -> {dates[-1].date()}")
px.tail(3).iloc[:, :6].round(2)

---
## 2 · 一只股票

<br>Understand **one** stock before looking at 109. Change `TICKER` and re-run.

In [ ]:
TICKER = "AAPL"

close = daily[TICKER].dropna()
rets  = close.pct_change(fill_method=None).dropna()   # 日收益 / daily returns

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6))
a.plot(close.index, close.values, color=tl.ACCENT, lw=1.1)
a.set_title(f"{TICKER} — daily close"); a.grid(alpha=.3)
b.hist(rets.values * 100, bins=80, color=tl.ACCENT, alpha=.85)
b.set_title(f"{TICKER} — daily returns (%)"); b.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"最好的一天 best day  : {rets.max():+.2%}  ({rets.idxmax().date()})")
print(f"最差的一天 worst day : {rets.min():+.2%}  ({rets.idxmin().date()})")
print("\n看那张直方图：绝大多数日子挤在 0 附近，两边远处各有几个巨大的日子。")
print("这些「肥尾」就是我们要用回撤和 VaR 衡量风险、而不是只看平均值的原因。")
print("Those fat tails are why we measure risk with drawdowns, not just an average.")
'''
how to describe  or summarize a set of data
data = (1,3213,432412,312.42,32,3423)
4 stats of data
1. mean, avg
2. std
3. skewness
4. kurtosis: fat tail
'''

---
## 3 · 五个指标

<br>The five numbers every fund reports. Each is three lines of arithmetic.

| | |
|---|---|
| **NAV** 净值 | 1 块钱变成了多少
| **CAGR** 年化 | 换算成「每年稳定涨多少」
| **Volatility** 波动率 | 路有多颠，×√12 换算成年
| **Sharpe** 夏普 | 每份风险换回多少收益
| **Max drawdown** 最大回撤 | 从高点摔得最惨的一次

<br>All Sharpes here assume **rf = 0**; proper excess-return Sharpes are ~0.2–0.3 lower.

In [ ]:
rows = {}
for t in ["SPY", "AAPL", "NVDA", "KO", "INTC"]:
    m = tl.metrics(tl.returns(px[t]).dropna())
    rows[t] = {"$1 ->": tl.nav(tl.returns(px[t]).dropna())[-1],
               "CAGR": m["cagr"], "vol": m["vol"],
               "Sharpe": m["sharpe"], "maxDD": m["max_drawdown"]}

table = pd.DataFrame(rows).T
display(table.style.format({"$1 ->": "{:.2f}x", "CAGR": "{:.1%}", "vol": "{:.1%}", "Sharpe": "{:.2f}", "maxDD": "{:.1%}"}))

print("NVDA 年化最高，但最大回撤足以让你在最低点割肉。")
print("SPY 的**夏普**赢过大多数个股，年化却输得很惨 —— 这就是分散化。")
print("SPY wins on SHARPE while losing badly on CAGR. That is diversification.")

---
## 4 · 策略就是一个「打分」


**The keystone.** A strategy is not a program, it is a **score** — one number per stock
per month. One engine runs them all without knowing what any of them mean.

In [ ]:
for s in STRATEGIES:
    print(f"  {s['id']:9} {s['sleeve']:11} {s['name']}")

# 12-1 动量：过去一年的收益，但**跳过最近一个月**。
# Momentum 12-1: last year's return, SKIPPING the most recent month.
#   px.iloc[i-1] / px.iloc[i-12] - 1
# 跳过是故意的：最近一个月往往会反转，算进去等于往动量里掺了反转。
# The skip is deliberate: the last month tends to REVERSE.
mom = BY_ID["MOM12_1"]

i = 60                                    # 中间某个月 / some month in the middle
scores = mom["score"](px, vol, i)
print(f"\n{dates[i].date()} 这个月，分数最高的 5 只 / top 5 that month:")
for t, v in scores.dropna().sort_values(ascending=False).head(5).items():
    print(f"   {t:6} {v:+.1%}")

---
## 5 · 回测 —— 看起来很赚钱


Each month: score → hold the top 10 equal-weight → pay cost on turnover → earn
**next** month's return.

In [ ]:
lo  = engine.backtest(px, vol, dates, mom["score"], mode="long_only")
spy = engine.benchmark_returns(px, dates)

lo_m, spy_m = tl.metrics(lo["net"]), tl.metrics(spy)
print(f"{'':12}{'$1 ->':>9}{'CAGR':>9}{'Sharpe':>9}{'maxDD':>9}{'beta':>8}")
print(f"{'策略':12}{tl.nav(lo['net'])[-1]:>8.2f}x{tl.pct(lo_m['cagr']):>9}"
      f"{lo_m['sharpe']:>9.2f}{tl.pct(lo_m['max_drawdown']):>9}{tl.beta_of(lo['net'], spy):>8.2f}")
print(f"{'SPY':12}{tl.nav(spy)[-1]:>8.2f}x{tl.pct(spy_m['cagr']):>9}"
      f"{spy_m['sharpe']:>9.2f}{tl.pct(spy_m['max_drawdown']):>9}{1.0:>8.2f}")

plt.plot(lo["dates"], tl.nav(lo["net"])[1:], color=tl.ACCENT, lw=1.6, label="Momentum 12-1")
plt.plot(lo["dates"], tl.nav(spy)[1:],       color=tl.GRAY,   lw=1.8, label="SPY")
plt.yscale("log"); plt.yticks([1, 2, 5, 10], ["$1", "$2", "$5", "$10"])
plt.title("Growth of $1, net of costs"); plt.legend(); plt.grid(alpha=.3); plt.show()

print("看起来像个天才选股者。接下来三格，我们把它拆掉。")
print("It looks like a brilliant stock picker. The next three cells take it apart.")

---
## 6 · 先插播：偷看未来

<br>The deadliest bug in backtesting is one character wide.

<br>The demo uses momentum *without* the skip, so its score contains this month's price.

In [ ]:
def backtest_offset(score_fn, offset):
    """offset=1 诚实 honest；offset=0 作弊 CHEATING。"""
    out, r = [], px.pct_change(fill_method=None)
    for i in range(config.WARMUP, len(dates) - 1):
        w   = engine.pick_weights(score_fn(px, vol, i), "long_only")
        nxt = r.iloc[i + offset]
        out.append(sum(x * nxt.get(t, 0.0) for t, x in w.items() if pd.notna(nxt.get(t))))
    return out

mom12_0 = lambda p, v, i: p.iloc[i] / p.iloc[i - 12] - 1     # 不跳过当月 / no skip

for label, off in [("offset=1  诚实 honest", 1), ("offset=0  作弊 CHEATING", 0)]:
    net = backtest_offset(mom12_0, off)
    m = tl.metrics(net)
    print(f"{label:26} Sharpe {m['sharpe']:5.2f}   $1 -> ${tl.nav(net)[-1]:,.2f}")

print("""
一个字符 —— 把 i+1 写成 i —— 把 12.79 块变成了 513.63 块。
One character turned $12.79 into $513.63.

为什么会炸：打分是 px[i]/px[i-12]-1，收益是 px[i]/px[i-1]-1，两者都含 px[i]。
所以「分数最高」基本等于「本月涨得最多」，然后我们再把本月的涨幅发给自己。
Both contain px[i], so "highest score" ~ "went up most THIS month".

回报仔细读代码的人：我们真正用的 MOM12_1 用的是 px[i-1]，天然对这个 bug 免疫。
Our real signal uses px[i-1] and is immune to exactly this bug.""")

---
## 7 · 把大盘拿掉


Buy the top tenth, **short** the bottom tenth, equal dollars. A rising market lifts the
longs and sinks the shorts, and they cancel. What is left is the signal, naked.

In [ ]:
ls = engine.backtest(px, vol, dates, mom["score"], mode="long_short", borrow_bps=0)
ls_m = tl.metrics(ls["net"])
lo_beta, ls_beta = tl.beta_of(lo["net"], spy), tl.beta_of(ls["net"], spy)

print(f"{'':12}{'CAGR':>9}{'Sharpe':>9}{'beta to SPY':>14}")
print(f"{'纯多头 LO':12}{tl.pct(lo_m['cagr']):>9}{lo_m['sharpe']:>9.2f}{lo_beta:>14.2f}")
print(f"{'多空  LS':12}{tl.pct(ls_m['cagr']):>9}{ls_m['sharpe']:>9.2f}{ls_beta:>14.2f}")

# 这张图是整个项目最有说服力的一张。每个点是一个月：横轴大盘、纵轴我们。
# 拟合直线的**斜率**就是 beta。
# The most convincing chart here. Each dot is a month; the SLOPE is the beta.
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
x = np.array(spy) * 100
for ax, (series, name, c) in zip(axes, [(lo["net"], "Long-only", tl.ACCENT),
                                        (ls["net"], "Long-short", tl.GOOD)]):
    y = np.array(series) * 100
    ax.scatter(x, y, s=14, color=c, alpha=.6)
    slope, b = np.polyfit(x, y, 1)
    ax.plot([x.min(), x.max()], [slope * x.min() + b, slope * x.max() + b], "k-", lw=1.2)
    ax.axhline(0, color="gray", lw=.6); ax.axvline(0, color="gray", lw=.6)
    ax.set_title(f"{name} — beta = {slope:.2f}"); ax.set_xlabel("SPY monthly return (%)")
    ax.grid(alpha=.3)
axes[0].set_ylabel("strategy monthly return (%)")
plt.tight_layout(); plt.show()

print(f"""
beta 从 {lo_beta:.2f} 变成 {ls_beta:.2f} —— 大盘几乎被完美拿掉，而且是免费的。
然后看收益：{tl.pct(lo_m['cagr'])} 塌成 {tl.pct(ls_m['cagr'])}。

**那消失的 23 个百分点，就是 beta。**
The 23 percentage points that vanished WERE the beta.""")

---
## 8 · 直接测信号


**IC (information coefficient)**: each month, rank stocks by score, rank them again by
what they actually did next month, and measure the agreement.

|---|---|
| 0.00–0.02 | 噪音
| 0.02–0.05 | 一个真实的股票信号


In [ ]:
'''
Quant reseach has stages
1. feature/signal/alpha
2. strategy/portfolio

'''



print(f"{'id':10}{'rank IC':>10}{'t':>8}   判定 verdict")
ic_all = {}
for s in STRATEGIES:
    ser = analysis.ic_series(px, vol, dates, s["score"], "rank")
    summ = analysis.ic_summary(ser)
    ic_all[s["id"]] = (ser, summ)
    print(f"{s['id']:10}{summ['mean']:>+10.4f}{summ['t']:>+8.2f}   "
          f"{'显著 significant' if abs(summ['t']) > 2 else '噪音 noise'}")

# 五分位：如果信号有效，这应该是一个**楼梯**。
# Quintiles: if the signal works, this should be a STAIRCASE.
buckets = analysis.quintile_buckets(px, vol, dates, mom["score"])
means = [np.mean(b) * 100 for b in buckets]

plt.bar([f"Q{i+1}" for i in range(5)], means,
        color=[tl.GOOD] + [tl.ACCENT] * 3 + [tl.BAD], alpha=.9)
plt.title("Avg next-month return by score bucket (Q1 = highest score)")
plt.ylabel("% per month"); plt.grid(alpha=.3, axis="y"); plt.show()

mono = all(means[i] >= means[i+1] for i in range(4))
print(f"是干净的楼梯吗 is it a staircase? {mono}")
print("""
不是。Q5（分数**最低**那批）跑赢了 Q2、Q3、Q4。
如果动量真的把这些股票排对了，这不可能发生。
Q5 — the LOWEST-scoring stocks — beat Q2, Q3 and Q4. If momentum really sorted
these stocks, that could not happen.

动量的 t = 0.39。「不是运气」的门槛大约是 2。念出来：
**我们无法排除「这个信号毫无价值」的可能性。**
We cannot reject the hypothesis that this signal is worthless.""")

---
## 9 · 那到底是什么？


Decompose the returns into known risk factors plus a leftover. The leftover is
**alpha** — the only honest claim to skill.

$$r = \alpha + \beta_1 \cdot \text{MKT} + \beta_2 \cdot \text{GROWTH} + \beta_3 \cdot \text{SIZE} + \beta_4 \cdot \text{RATES} + \beta_5 \cdot \text{GOLD} + \beta_6 \cdot \text{USD} + \varepsilon$$

<br>The least squares is hand-written in `lib/analysis.py`, not a library call.

In [ ]:
mr = px.pct_change(fill_method=None)
F  = analysis.build_factors(mr, dates)

for name, series in [("纯多头 LONG-ONLY", lo["net"]), ("多空 LONG-SHORT", ls["net"])]:
    fit = analysis.ols(series, F)
    print(f"\n=== {name}   R² = {fit['r2']:.3f} ===")
    print(f"{'':9}{'coef':>9}{'std err':>10}{'t':>8}")
    for j, n in enumerate(fit["names"]):
        star = " *" if abs(fit["t"][j]) > 2 else ""
        val  = f"{fit['beta'][j]*12:>8.2%}" if n == "ALPHA" else f"{fit['beta'][j]:>9.3f}"
        print(f"{n:9}{val}{fit['se'][j]:>10.3f}{fit['t'][j]:>8.2f}{star}")

print("""
把两张表对着读：

纯多头 R² = 0.65 —— 它做的事情有三分之二只是在扛已知风险，主要是 1.01 的市场 beta。
它不是选股者，它是带点倾斜的大盘。

多空组合确实市场中性了（beta ≈ −0.08），但它在 GROWTH 上载荷 +0.43、t = 2.7。
它根本不「中性」—— 它是一个减掉了大盘的、加杠杆的成长股押注。
而它的 alpha，t = 0.37，和 0 没有区别。

The long-short book is market-neutral but NOT neutral: it is a leveraged growth bet
with the market subtracted, and its alpha is indistinguishable from zero.""")

---
## 结论

<br>**Three independent methods agree:**

| 方法
|---|---|

> <br>The apparent edge was **market beta and a growth tilt**, not stock selection.

<br>The one number that survives scrutiny is the **portfolio information ratio of 0.78**.

---


- 完整报告
- 每个指标怎么算
- 自己动手改
- 一步步学

### 想自己试试？


```python
my_score = lambda p, v, i: p.iloc[i]
engine.backtest(px, vol, dates, my_score, mode="long_short")
```

<br>Then ask the real question: **is its IC t-statistic above 2?**